# Video Object Removal & Inpainting — Demo

AIAA 3201 Project 3 — Spring 2026

This notebook demonstrates the three-part pipeline:
1. **Part 1 (Baseline)**: YOLOv8-Seg + Optical Flow + cv2.inpaint
2. **Part 2 (SOTA)**: SAM 2 / Tracker + ProPainter
3. **Part 3 (Exploration)**: Mask refinement + Diffusion inpainting

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

from utils.video_io import extract_frames, frames_to_video, get_video_info
from utils.mask_utils import save_masks, load_masks

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 8)

## Setup — Set your video path

In [ ]:
VIDEO_PATH = '../data/sample/your_video.mp4'  # <-- Change this

info = get_video_info(VIDEO_PATH)
print(f"Video: {VIDEO_PATH}")
print(f"Resolution: {info['width']}x{info['height']}")
print(f"FPS: {info['fps']:.1f}, Frames: {info['frame_count']}")

frames = extract_frames(VIDEO_PATH)
print(f"Loaded {len(frames)} frames")

## Part 1 — Baseline Pipeline

In [ ]:
from part1_baseline.detect_and_segment import load_model, detect_video
from part1_baseline.optical_flow import filter_masks_by_flow
from part1_baseline.inpaint_cv2 import inpaint_video

# Detection
model = load_model('yolov8m-seg.pt')
raw_masks = detect_video(model, frames)
print(f"Detected objects in {sum(1 for m in raw_masks if m.max() > 0)}/{len(raw_masks)} frames")

# Optical flow filtering
refined_masks = filter_masks_by_flow(frames, raw_masks)

# Inpainting
part1_result = inpaint_video(frames, refined_masks)

In [ ]:
# Visualize Part 1 results
sample_idx = len(frames) // 2
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(frames[sample_idx], cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[1].imshow(refined_masks[sample_idx], cmap='gray')
axes[1].set_title('Mask (after flow filtering)')
axes[2].imshow(cv2.cvtColor(part1_result[sample_idx], cv2.COLOR_BGR2RGB))
axes[2].set_title('Part 1 Inpainted')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Part 2 — SOTA Pipeline

In [ ]:
import os, tempfile, cv2
from part2_sota.sam2_tracker import get_tracker, SAM2Tracker, FallbackTracker
from part2_sota.propainter_inpaint import get_inpainter
from part1_baseline.detect_and_segment import load_model, detect_frame

tracker = get_tracker(use_sam2=True)

if isinstance(tracker, FallbackTracker):
    part2_masks = tracker.track(frames)
else:
    print('SAM 2 available — running video-level tracking')
    tmp_dir = tempfile.mkdtemp(prefix='sam2_frames_')
    for i, f in enumerate(frames):
        cv2.imwrite(os.path.join(tmp_dir, f'{i:05d}.jpg'), f, [cv2.IMWRITE_JPEG_QUALITY, 95])
    yolo = load_model('yolov8m-seg.pt')
    init_mask, _ = detect_frame(yolo, frames[0])
    part2_masks = tracker.track_from_detections(tmp_dir, [init_mask], [0])

inpainter = get_inpainter(use_propainter=True)
part2_result = inpainter.inpaint(frames, part2_masks)

## Part 3 — Exploration

In [ ]:
from part3_exploration.sam3_upgrade import get_mask_refiner
from part3_exploration.diffusion_inpaint import get_diffusion_inpainter

# Mask refinement
refiner = get_mask_refiner()
part3_masks = refiner.refine_masks(frames, part2_masks)

# Advanced inpainting
adv_inpainter = get_diffusion_inpainter()
part3_result = adv_inpainter.inpaint(frames, part3_masks)

## Comparison & Evaluation

In [ ]:
from evaluation.visualize import save_comparison_grid

results_dict = {
    'Part 1 (Baseline)': part1_result,
    'Part 2 (SOTA)': part2_result,
    'Part 3 (Exploration)': part3_result,
}

save_comparison_grid(
    frames, refined_masks, results_dict,
    output_path='../results/comparison_grid.png',
    max_frames=4,
)

# Display
from IPython.display import Image
Image('../results/comparison_grid.png')

In [ ]:
# Save output videos
fps = info['fps']
frames_to_video(part1_result, '../results/part1/inpainted.mp4', fps=fps)
frames_to_video(part2_result, '../results/part2/inpainted.mp4', fps=fps)
frames_to_video(part3_result, '../results/part3/inpainted.mp4', fps=fps)
print('All videos saved!')